# DroneWatch — İHA Tespit ve Takip Sistemi

Bu notebook, YOLO11 tabanlı bir nesne tespit modelini kullanarak video akışında **drone/İHA** tespiti ve takibi yapar.

**Neler yapacağız:**
1. Ortamı kur
2. Veri setini indir (Roboflow)
3. Modeli eğit
4. Sonuçları değerlendir
5. Video üzerinde tespit + takip çalıştır
6. (Opsiyonel) Yoğunluk haritası (heatmap) üret

> Yazar: *(kendi adınızı buraya ekleyin)*  
> Tarih: *(proje tarihi)*

## 1. Ortam Kurulumu

In [ ]:
!pip install ultralytics roboflow -q

import os
import cv2
from ultralytics import YOLO

print("Kurulum tamamlandı.")

## 2. Veri Setini İndirme

Roboflow üzerinden `Drones Yolo11 A` veri setini kullanıyoruz (9.900 görüntü, 5 sınıf: drone, kuş, uçak, vb.).
Kendi Roboflow API anahtarınızı [buradan](https://app.roboflow.com/settings/api) alıp aşağıya girin.

İsterseniz kendi topladığınız görüntülerle de veri setini genişletebilirsiniz.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "BURAYA_KENDI_API_ANAHTARINIZI_GIRIN"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("drone-a7lpy").project("drones-yolo11-a")
dataset_version = project.version(1)
dataset = dataset_version.download("yolov11")

DATASET_DIR = dataset.location
print("Veri seti indirildi:", DATASET_DIR)

## 3. Model Eğitimi

`yolo11n` (nano) ile hızlı bir baseline eğitelim. Daha yüksek doğruluk isterseniz `yolo11s` veya `yolo11m` deneyebilirsiniz — eğitim süresi de artar.

In [ ]:
base_model = YOLO("yolo11n.pt")

train_run = base_model.train(
    data=f"{DATASET_DIR}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    patience=10,
    project="drone_watch_runs",
    name="train_v1"
)

## 4. Değerlendirme

Eğitim sonrası doğrulama setinde metrikleri kontrol edelim (precision, recall, mAP).

In [ ]:
best_weights_path = "drone_watch_runs/train_v1/weights/best.pt"
trained_model = YOLO(best_weights_path)

metrics = trained_model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## 5. Video Üzerinde Tespit + Takip

Ultralytics'in yerleşik `track()` fonksiyonu ile ByteTrack algoritmasını kullanarak drone'ları kareler arasında takip ediyoruz.

Kendi test videonuzu Colab'a yükleyip `INPUT_VIDEO` yolunu güncelleyin.

In [ ]:
INPUT_VIDEO = "/content/test_video.mp4"

tracking_results = trained_model.track(
    source=INPUT_VIDEO,
    conf=0.35,
    iou=0.5,
    tracker="bytetrack.yaml",
    save=True,
    project="drone_watch_runs",
    name="track_v1"
)

print("Takip tamamlandı. Çıktı klasörü: drone_watch_runs/track_v1")

## 6. (Opsiyonel) Yoğunluk Haritası

Zaman içinde tespitlerin kümelendiği bölgeleri görselleştirmek için basit bir yoğunluk haritası oluşturuyoruz.
Bu, olası uçuş rotalarını veya sık görülen alanları anlamak için kullanılabilir.

In [ ]:
import numpy as np

def build_density_map(video_path, model, conf_threshold=0.35):
    cap = cv2.VideoCapture(video_path)
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    density_grid = np.zeros((frame_h, frame_w), dtype=np.float32)

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        detections = model.predict(frame, conf=conf_threshold, verbose=False)[0]
        for box in detections.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = box.astype(int)
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            cv2.circle(density_grid, (cx, cy), radius=25, color=1, thickness=-1)

    cap.release()
    return density_grid

density_grid = build_density_map(INPUT_VIDEO, trained_model)

heatmap_normalized = cv2.normalize(density_grid, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
heatmap_colored = cv2.applyColorMap(heatmap_normalized, cv2.COLORMAP_JET)
cv2.imwrite("drone_watch_runs/density_heatmap.png", heatmap_colored)

print("Yoğunluk haritası kaydedildi: drone_watch_runs/density_heatmap.png")

## Sonraki Adımlar

- [ ] Daha büyük/çeşitli bir veri seti ile yeniden eğitim
- [ ] Farklı model boyutlarını (`s`, `m`, `l`) karşılaştırma
- [ ] Gerçek zamanlı webcam desteği ekleme
- [ ] Basit bir Streamlit arayüzü ile sonuçları görselleştirme
- [ ] Sonuçları README'de belgeleme (kendi bulgularınız, grafikleriniz)